In [71]:
from playwright.async_api import async_playwright
from pathlib import Path
from bs4 import BeautifulSoup
from lxml import html
import re
import asyncio
from pydantic import BaseModel

In [42]:
auth_dir = Path("playwright", ".auth")

results_count_xpath = "/html/body/div[1]/div[1]/div/main/div/div/h1"
cards_layout_xpath = "/html/body/div[1]/div[1]/div/main/div/div/div/div[1]"

In [64]:
async with async_playwright() as p:
    browser = await p.chromium.launch(headless=False)
    context = await browser.new_context(storage_state=auth_dir / "magnit.json")
    page = await context.new_page()
    search_url = "https://magnit.ru/search?term=сыр"
    await page.goto(search_url, wait_until="domcontentloaded", timeout=60000)
    await page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
    content = await page.content()
    soup = BeautifulSoup(content, "html.parser")

    with open("magnit.html", "w", encoding="utf-8") as f:
        f.write(soup.prettify())

    

In [65]:
with open("magnit.html", "r", encoding="utf-8") as f:
    html_magnit = f.read()

tree = html.fromstring(bytes(html_magnit, encoding="utf-8"))

In [72]:
cards_layout = tree.xpath(cards_layout_xpath)[0]

cards = cards_layout.xpath(".//article")
print(f"Найдено карточек: {len(cards)}")

EXTRACT_RATIO = 0.35
need_to_extract = min(int(len(cards) * EXTRACT_RATIO), 20)

class Product(BaseModel):
    title: str
    price: str
    image_url: str
    product_url: str
    kkal: str = None
    protein: str = None
    fats: str = None
    carbs: str = None

products = []

for i, card in enumerate(cards[:need_to_extract]):
    title_el = card.xpath("./a/div[2]/div[1]/div[2]")
    price_el = card.xpath("./a/div[2]/div[1]/div[1]/div/span")
    image_el = card.xpath("./a/div[1]/div/img")
    href_el = card.xpath("./a/@href")
    
    title = title_el[0].text_content().strip() if title_el else "Без названия"
    price_text = price_el[0].text_content().strip() if price_el else "Цена не указана"
    image_src = image_el[0].get("src") if image_el else "Изображение не найдено"
    href = href_el[0] if href_el else "Ссылка не найдена"

    product = Product(
        title=title,
        price=price_text,
        image_url=image_src,
        product_url=f"https://magnit.ru{href}"
    )
    print(f"{i+1}: {product.title} - {product.price} - {product.product_url}")
    products.append(product)

Найдено карточек: 36
1: Сыр творожный Violette с креветками 70% 140г - 99.99 ₽ - https://magnit.ru/product/1000122984-violette_syr_tvorozhnyy_s_krevetkami70_140g_pl_st_karat_12?shopCode=784507&shopType=express
2: Сыр Сметанковый Варвара Краса 50% 160г - 149.99 ₽ - https://magnit.ru/product/1000383360-varvara_krasa_syr_smetankovyy_50_160g?shopCode=784507&shopType=express
3: Сыр Сливочный Вкуснотеево 45% 300г - 249.99 ₽ - https://magnit.ru/product/8000060200-vkusnoteevo_syr_slivochnyy_45_300g?shopCode=784507&shopType=express
4: Сыр Брест-Литовск Голландский 45% 200г - 189.99 ₽ - https://magnit.ru/product/1000316963-syr_golandskiy_brest_litovsk_45_200g?shopCode=784507&shopType=express
5: Сыр Ламбер 50% весовой - 324.97 ₽ - https://magnit.ru/product/1000074377-lamber_syr_50_f_vbd_2_5?shopCode=784507&shopType=express
6: Сыр Поставы Городок Пармезан гранд 45% 200г - 170 ₽ - https://magnit.ru/product/1000125628-syr_parmezan_postavy_gorodok_grand_45_200g?shopCode=784507&shopType=express
7: Тер

In [ ]:
async with async_playwright() as p:
    browser = await p.chromium.launch(headless=False)
    context = await browser.new_context(storage_state=auth_dir / "magnit.json")
    page = await context.new_page()
    for product in products:
        await page.goto(product.product_url, wait_until="domcontentloaded", timeout=60000)
        await page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
        content = await page.content()
        soup = BeautifulSoup(content, "html.parser")

        product_tree = html.fromstring(bytes(str(soup), encoding="utf-8"))
        
        kkal_el = product_tree.xpath("/html/body/div[2]/div[1]/div/div/main/div/div/div[1]/div/div/div/div[1]/div[2]/section[2]/div[2]/div[1]/div[2]")
        protein_el = product_tree.xpath("/html/body/div[2]/div[1]/div/div/main/div/div/div[1]/div/div/div/div[1]/div[2]/section[2]/div[2]/div[2]/div[2]")
        fats_el = product_tree.xpath("/html/body/div[2]/div[1]/div/div/main/div/div/div[1]/div/div/div/div[1]/div[2]/section[2]/div[2]/div[3]/div[2]")
        carbs_el = product_tree.xpath("/html/body/div[2]/div[1]/div/div/main/div/div/div[1]/div/div/div/div[1]/div[2]/section[2]/div[2]/div[4]/div[2]")

        kkal = kkal_el[0].text_content().strip() if len(kkal_el) > 0 else "Калории не найдены"
        protein = protein_el[0].text_content().strip() if len(protein_el) > 0 else "Белки не найдены"
        fats = fats_el[0].text_content().strip() if len(fats_el) > 0 else "Жиры не найдены"
        carbs = carbs_el[0].text_content().strip() if len(carbs_el) > 0 else "Углеводы не найдены"

        product.kkal = kkal
        product.protein = protein
        product.fats = fats
        product.carbs = carbs

In [76]:
for product in products:
    print(f"{product.title} - {product.price} - {product.kkal} - {product.protein} - {product.fats} - {product.carbs} - {product.product_url}")

Сыр творожный Violette с креветками 70% 140г - 99.99 ₽ - 266 - 7 - 25 - 3.3 - https://magnit.ru/product/1000122984-violette_syr_tvorozhnyy_s_krevetkami70_140g_pl_st_karat_12?shopCode=784507&shopType=express
Сыр Сметанковый Варвара Краса 50% 160г - 149.99 ₽ - 356 - 26 - 28 - 0 - https://magnit.ru/product/1000383360-varvara_krasa_syr_smetankovyy_50_160g?shopCode=784507&shopType=express
Сыр Сливочный Вкуснотеево 45% 300г - 249.99 ₽ - 320 - 24 - 25 - Углеводы не найдены - https://magnit.ru/product/8000060200-vkusnoteevo_syr_slivochnyy_45_300g?shopCode=784507&shopType=express
Сыр Брест-Литовск Голландский 45% 200г - 189.99 ₽ - 332 - 26.3 - 25.2 - Углеводы не найдены - https://magnit.ru/product/1000316963-syr_golandskiy_brest_litovsk_45_200g?shopCode=784507&shopType=express
Сыр Ламбер 50% весовой - 324.97 ₽ - 365 - 26 - 30 - Углеводы не найдены - https://magnit.ru/product/1000074377-lamber_syr_50_f_vbd_2_5?shopCode=784507&shopType=express
Сыр Поставы Городок Пармезан гранд 45% 200г - 170 ₽ -